# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Devaaldo/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook audits the production feature vector for content decay prediction, verifying zero label leakage, temporal feasibility, and client privacy boundaries.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We build a 13-feature continuous feature vector ($X$) from the active content snapshot (`content_refresh_anonymized.csv`). Traffic count metrics are transformed using $\log(1 + x)$ to normalize heavy-tailed distributions. Missing values are imputed with 0.0.

In [1]:
# 1. Load Data and Construct Feature Vector
from pathlib import Path
import numpy as np
import pandas as pd

curr = Path.cwd()
candidates = [curr, curr.parent.parent, curr.parent]
data_path = None
for c in candidates:
    p = c / "data/raw/content_refresh_anonymized.csv"
    if p.exists():
        data_path = p
        break
if data_path is None:
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Feature Engineering: Log-transformed counts
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].fillna(0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].fillna(0))
df["log_sessions_90d"] = np.log1p(df["sessions_90d"].fillna(0))

FEATURE_COLS = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "days_since_last_update", "content_age_days",
    "days_with_impressions", "days_with_sessions",
    "word_count", "char_count"
]

for col in FEATURE_COLS:
    df[col] = df[col].fillna(0.0)

X = df[FEATURE_COLS]
y = df["is_declining_label"]

print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Vector Shape  : {y.shape}")
print(f"Total Unique Clients : {df['client_id'].nunique()}")
print(f"Dataset Base Rate    : {y.mean():.1%}")

Feature Matrix Shape : (30000, 13)
Target Vector Shape  : (30000,)
Total Unique Clients : 32
Dataset Base Rate    : 54.2%


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every feature in the matrix represents an observational metric knowable *at the moment of weekly triage review*. No future-looking or outcome-derived metrics are included.

In [2]:
# 2. Audit Feature Metadata and Missingness Summary
feature_audit = []
for col in FEATURE_COLS:
    feature_audit.append({
        "Feature": col,
        "Dtype": str(df[col].dtype),
        "Missing %": f"{df[col].isna().mean():.1%}",
        "Mean": round(float(df[col].mean()), 2),
        "Min": round(float(df[col].min()), 2),
        "Max": round(float(df[col].max()), 2)
    })

audit_df = pd.DataFrame(feature_audit)
print("=" * 75)
print("FEATURE METADATA & INTEGRITY PROFILE")
print("=" * 75)
print(audit_df.to_string(index=False))

FEATURE METADATA & INTEGRITY PROFILE
               Feature   Dtype Missing %     Mean   Min       Max
   log_impressions_90d float64      0.0%     6.19  0.69     13.16
        log_clicks_90d float64      0.0%     1.21  0.00      8.34
      log_sessions_90d float64      0.0%     2.41  0.69      8.38
          avg_position float64      0.0%    16.34  0.00    245.00
                   ctr float64      0.0%     0.51  0.00    100.00
       engagement_rate float64      0.0%     2.53  0.00    100.00
           scroll_rate float64      0.0%    18.14  0.00    300.00
days_since_last_update   int64      0.0%    46.10  1.00    373.00
      content_age_days   int64      0.0%   256.17 90.00    564.00
 days_with_impressions   int64      0.0%    61.45  1.00     88.00
    days_with_sessions   int64      0.0%    13.10  1.00     90.00
            word_count float64      0.0%  2310.21  0.00   9546.00
            char_count float64      0.0% 15361.88  0.00 111158.00


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We test feature correlations with the target and execute the **Confession Test** to verify that our evaluation harness detects leakage.

In [3]:
# 3. Correlation Check & Confession Test
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

corrs = df[FEATURE_COLS].apply(lambda x: x.corr(df["is_declining_label"])).round(3)
print("Feature Correlations with is_declining_label:")
print(corrs.sort_values(ascending=False).to_string())

# Confession Test: Adding trend_pct directly
lr_clean = LogisticRegression(max_iter=500, random_state=42).fit(X, y)
auc_clean = roc_auc_score(y, lr_clean.predict_proba(X)[:, 1])

X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"].fillna(0.0)
lr_leaky = LogisticRegression(max_iter=500, random_state=42).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, lr_leaky.predict_proba(X_leaky)[:, 1])

print("\n" + "=" * 60)
print("CONFESSION TEST VERIFICATION")
print("=" * 60)
print(f"Clean Features ROC-AUC : {auc_clean:.3f} (Realistic, Honest)")
print(f"Leaky Features ROC-AUC : {auc_leaky:.3f} (Near-perfect cheating)")
print("Verdict: Target leakage caught and excluded from production matrix.")

Feature Correlations with is_declining_label:
days_with_impressions     0.190
log_impressions_90d       0.177
word_count                0.119
char_count                0.108
days_since_last_update    0.081
log_sessions_90d          0.015
log_clicks_90d            0.003
scroll_rate              -0.003
engagement_rate          -0.013
days_with_sessions       -0.025
avg_position             -0.029
ctr                      -0.062
content_age_days         -0.164


C:\Users\akbarprdna\AppData\Local\Temp\flyrank-starter\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



CONFESSION TEST VERIFICATION
Clean Features ROC-AUC : 0.689 (Realistic, Honest)
Leaky Features ROC-AUC : 0.999 (Near-perfect cheating)
Verdict: Target leakage caught and excluded from production matrix.


C:\Users\akbarprdna\AppData\Local\Temp\flyrank-starter\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Every excluded column is documented with explicit methodological justification.

In [4]:
# 4. Excluded Columns Documentation Table
exclusions = [
    {"Column": "trend_direction", "Reason": "Directly generates the target label (is_declining_label == down). Fatal leakage."},
    {"Column": "trend_pct", "Reason": "Mathematical source of trend_direction. Sibling leakage."},
    {"Column": "client_id", "Reason": "Categorical tenant ID. Used only for grouped holdout splitting to prevent domain memorization."},
    {"Column": "content_id", "Reason": "Unique row identifier. Carries no generalizable predictive value."},
    {"Column": "flag_fix_ctr", "Reason": "Pre-existing rule heuristic flag. Circular decision leakage."},
    {"Column": "flag_zombie_page", "Reason": "Existing heuristic status tag. Circular decision leakage."}
]
excl_df = pd.DataFrame(exclusions)
print("=" * 85)
print("EXCLUDED ATTRIBUTES & METHODOLOGICAL JUSTIFICATION")
print("=" * 85)
print(excl_df.to_string(index=False))

EXCLUDED ATTRIBUTES & METHODOLOGICAL JUSTIFICATION
          Column                                                                                         Reason
 trend_direction               Directly generates the target label (is_declining_label == down). Fatal leakage.
       trend_pct                                       Mathematical source of trend_direction. Sibling leakage.
       client_id Categorical tenant ID. Used only for grouped holdout splitting to prevent domain memorization.
      content_id                              Unique row identifier. Carries no generalizable predictive value.
    flag_fix_ctr                                   Pre-existing rule heuristic flag. Circular decision leakage.
flag_zombie_page                                      Existing heuristic status tag. Circular decision leakage.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.